# 🏆 Customer Lifetime Value (CLTV) Prediction System
## Phase 1: Business Understanding
**Project**: Fortune 500 E-Commerce CLTV Prediction  
**Author**: Nisarga N  
**Date**: June 2026  
**Dataset**: UCI Online Retail II (UK-based online retailer, 2009–2011)  
**Objective**: Understand the business problem of CLTV prediction before writing a single line of ML code.

---
> **Interview Tip**: In every data science interview, the first question is always: *"Walk me through your thought process."* This notebook demonstrates that you start with business understanding — exactly as the CRISP-DM and TDSP methodologies recommend.


## 1. What is Customer Lifetime Value (CLTV)?

Customer Lifetime Value is **the total net revenue a business can expect from a single customer account throughout the entire business relationship**.

### 📐 The Core Formula

$$CLTV = \frac{Average\ Purchase\ Value \times Purchase\ Frequency}{Churn\ Rate} \times Profit\ Margin$$

Or in its expanded predictive form:
$$CLTV = \sum_{t=1}^{T} \frac{Revenue_t - Cost_t}{(1 + d)^t} \times Survival\ Probability_t$$

Where:
- **T** = prediction horizon (e.g., 12 months)
- **d** = discount rate (cost of capital, typically 10%)
- **Survival Probability** = probability the customer is still active at time t

### 🔑 Key Distinction: Historical vs Predictive CLTV
| Approach | Definition | Use Case | Limitation |
|----------|------------|----------|------------|
| **Historical CLTV** | Sum of all past revenue from a customer | Customer ranking, reporting | Backward-looking |
| **Predictive CLTV** | ML/statistical forecast of future revenue | Budget allocation, targeting | Requires enough history |
- **Traditional Formula** | AOV × Frequency × Lifespan | Quick estimates | Oversimplifies behaviour |

> **Interview Answer**: "Historical CLTV tells you who was valuable. Predictive CLTV tells you who WILL BE valuable. For marketing decisions, you need the latter."


In [ ]:
# Standard library imports
import json
import warnings
from pathlib import Path

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# ─── Plot style configuration ───────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'figure.titlesize': 16,
})

# Color palette for the project (consistent across all notebooks)
COLOR_PALETTE = {
    'primary': '#58a6ff',
    'secondary': '#3fb950',
    'accent': '#d2a8ff',
    'warning': '#f0883e',
    'danger': '#f85149',
    'success': '#56d364',
    'neutral': '#8b949e',
    'gold': '#e3b341',
    'gradient': ['#58a6ff', '#3fb950', '#d2a8ff', '#f0883e', '#f85149'],
}

print('✅ Environment configured successfully')
print(f'📊 Plotting style: Dark mode (GitHub-inspired)')
print(f'🎨 Project color palette: {list(COLOR_PALETTE.keys())}')


## 2. Why Do Fortune 500 Companies Predict CLTV?

Write a comprehensive markdown cell explaining:
- Amazon uses CLTV to decide how much to spend on Prime member acquisition (they reportedly spend $300+ to acquire a member worth $2,500 CLTV)
- Netflix uses CLTV to decide which content to produce — shows watched by high-CLTV subscribers get renewed
- Spotify segments users by predicted LTV to target Spotify Premium upsell campaigns
- Uber uses CLTV to determine driver supply in markets with high-LTV rider density
- Swiggy/Zomato use CLTV to decide which restaurants to onboard and which cities to expand to
- Flipkart uses CLTV to determine cashback amounts on credit card partnerships

Include a table: Company | Industry | CLTV Use Case | Business Impact


In [ ]:
# Create visualization showing why customer segments matter
segments = ['New Customers', 'Regular Shoppers', 'Loyal Customers', 'Champions']
cac = [50, 50, 50, 50]
ltv = [120, 380, 850, 2200]
ratios = ['2.4x', '7.6x', '17x', '44x']

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(segments))
width = 0.35

rects1 = ax.bar(x - width/2, cac, width, label='CAC (Customer Acquisition Cost)', color=COLOR_PALETTE['danger'])
rects2 = ax.bar(x + width/2, ltv, width, label='LTV (Lifetime Value)', color=COLOR_PALETTE['secondary'])

ax.set_ylabel('Value (£)')
ax.set_title('CLTV vs CAC: Why Customer Segments Matter')
ax.set_xticks(x)
ax.set_xticklabels(segments)
ax.legend()

# Add ratio labels
for i, rect in enumerate(rects2):
    height = rect.get_height()
    ax.annotate(f'Ratio: {ratios[i]}',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontweight='bold', color=COLOR_PALETTE['primary'])

# Save path
viz_dir = Path.cwd().parent / "visualizations" / "eda"
viz_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(viz_dir / "01_ltv_vs_cac.png", dpi=150, facecolor='#0d1117')
plt.show()


### 💡 Business Insight
A Champion customer generates 44x their acquisition cost in value, whereas a New customer generates only 2.4x. This indicates that retention efforts focused on converting Regular Shoppers to Loyal and Champions yield exponential ROI compared to general acquisition.
